# Merge Linguistic Proximity (prox1) Country-Pair Lookup

Reads the raw Melitz & Toubal (2014) `prox1` table
(`data/gravity/melitz_toubal_proxling.dta` -- the same source Bekes and Ottaviano 2025
use for their own language-similarity variable) and resolves it into a complete,
closed country-pair lookup over every ISO3 code that appears anywhere in our match
data: self-pairs set to 1.0, Monaco (absent from the raw table) resolved via the
same fallback rule used everywhere else in this project (Monaco vs. a
French-official country -> 1.0, Monaco vs. anyone else -> France's value as a
stand-in). Belgium is also entirely absent from the raw table (fixed 2026-09-12,
previously undetected -- every BEL pair silently defaulted to 0.0) and is resolved
as a population-weighted blend of NLD's and FRA's own values (Dutch ~59%, French
~41%), mirroring how Melitz & Toubal (2014) themselves treat 2-language countries
like Switzerland. Any other missing pair (mostly involving South Korea, entirely
absent from this table's country coverage) falls back to 0.

**Provenance column (added 2026-09-12)**: every row in the output now carries a `note`
field -- empty ('') when the value comes directly from the raw Melitz-Toubal table with
no adjustment, otherwise a short description of which fallback rule produced it (Monaco
stand-in, Belgium population-weighted blend, or generic missing-pair-defaults-to-0). This
makes every imputed/adjusted number traceable back to its origin instead of looking
indistinguishable from genuine raw source data.

**Output**: `data/gravity/ling_prox_pairs_final.csv` -- a complete lookup requiring zero
further fallback logic downstream. `homophily.ipynb` reads this file directly for
Section 6/6.1/6.2's random-matching benchmark and field-composition calculations
(which query arbitrary same-tournament country pairs, not just pairs realized as
actual matches, so this can't be pre-baked into the match-level panel the way
`winners/losers_ling_prox_cont` already is upstream in `final_ds.ipynb`).

**Pipeline position**: independent of `final_ds.ipynb`/`homophily.ipynb` -- run
whenever `melitz_toubal_proxling.dta` changes or the country universe in the match
data expands. Does not touch `men_matches_with_ranks_cleaned.xlsx`, `team_gs_panel.csv`,
or `tiebreak_panel.csv` -- those already carry the per-match `prox1` columns via
`final_ds.ipynb` (see the warning cell there about not re-running it from raw inputs).

In [1]:
import os
import pandas as pd
from itertools import combinations

ROOT       = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
EXCEL_PATH = os.path.join(ROOT, 'data', 'atp', 'men_matches_with_ranks_cleaned.xlsx')
PROX_PATH  = os.path.join(ROOT, 'data', 'gravity', 'melitz_toubal_proxling.dta')
OUT_PATH   = os.path.join(ROOT, 'data', 'gravity', 'ling_prox_pairs_final.csv')
print('Paths set.')

Paths set.


## 1. Country universe: every ISO3 appearing in our match data

In [2]:
iso_cols = ['winners_p1_iso3', 'winners_p2_iso3', 'losers_p1_iso3', 'losers_p2_iso3']
raw = pd.read_excel(EXCEL_PATH, sheet_name='players_list', usecols=iso_cols)

isos = set()
for c in iso_cols:
    isos |= set(raw[c].dropna().unique())
isos = sorted(isos)
print(f'Country universe: {len(isos)} ISO3 codes')

Country universe: 64 ISO3 codes


## 2. Raw prox1 lookup + Monaco/KOR fallback resolution

In [3]:
prox_raw = pd.read_stata(PROX_PATH)
raw_lut = {}
for _, r in prox_raw.iterrows():
    if pd.isna(r['iso3_o']) or pd.isna(r['iso3_d']) or pd.isna(r['proxling']):
        continue
    raw_lut[tuple(sorted([r['iso3_o'], r['iso3_d']]))] = float(r['proxling'])
print(f'Raw Melitz-Toubal country-pairs: {len(raw_lut):,}')

FRENCH_OFFICIAL = {'FRA', 'BEL', 'CHE', 'CAN', 'LUX'}

# Belgium (BEL) fix (2026-09-12): BEL is entirely absent from the raw Melitz-Toubal
# table -- the same problem Monaco has -- but had no fallback until now, so every
# BEL pair (22 match-team-obs in our data) silently fell to the generic 0.0 default,
# scoring e.g. BEL-FRA as maximally distant despite Belgium being ~40% French-speaking.
# Fix: treat Belgium as a 2-language country exactly like Melitz & Toubal (2014) treat
# Switzerland (German 0.74 / French 0.26) -- population-weighted blend of Dutch and
# French, using NLD and FRA as proxy countries for each language, weighted by Belgium's
# own official-language population shares (Dutch ~59%, French ~41%; the ~1% German-
# speaking minority is dropped and the remainder rescaled to sum to 1, mirroring how
# Melitz & Toubal cap every country at its top 2 native languages).
BELGIUM_LANG_WEIGHTS = {'NLD': 0.59, 'FRA': 0.41}

def _raw_prox(x, y):
    """proxling(x, y), treating x==y as the trivial same-language case (1.0) since
    self-pairs are not always present as explicit rows in the raw table."""
    if x == y:
        return 1.0
    return raw_lut.get(tuple(sorted([x, y])))

def resolve_prox1(a, b):
    """Returns (value, note). note is '' when the value comes straight from the raw
    Melitz-Toubal table with no adjustment; otherwise it records which fallback rule
    produced the value, so ling_prox_pairs_final.csv can flag every imputed/adjusted
    number rather than presenting it as indistinguishable from the raw source data.

    Rules: MCO (absent from the raw table) vs. a French-official country -> 1.0; MCO
    vs. anyone else -> France's value as a stand-in. BEL (also absent) -> population-
    weighted blend of NLD's and FRA's own values against the other country (see
    BELGIUM_LANG_WEIGHTS above). Any other missing pair (mostly KOR, entirely absent
    from this 1990s-2000s country coverage) falls back to 0."""
    if a == b:
        return 1.0, ''
    if a == 'MCO' or b == 'MCO':
        other = b if a == 'MCO' else a
        if other in FRENCH_OFFICIAL:
            return 1.0, 'imputed: Monaco set to 1.0 (French-official country pair)'
        val = raw_lut.get(tuple(sorted(['FRA', other])))
        if val is None:
            return 0.0, 'imputed: Monaco fallback -- France\'s own value also missing, defaulted to 0'
        return val, 'imputed: Monaco value stands in for France (Monaco absent from raw table)'
    if a == 'BEL' or b == 'BEL':
        other = b if a == 'BEL' else a
        parts = [(_raw_prox(lang, other), w) for lang, w in BELGIUM_LANG_WEIGHTS.items()]
        parts = [(v, w) for v, w in parts if v is not None]
        if not parts:
            return 0.0, 'imputed: Belgium fallback -- NLD/FRA values also missing, defaulted to 0'
        total_w = sum(w for _, w in parts)
        blended = sum(v * w for v, w in parts) / total_w
        return blended, 'imputed: Belgium absent from raw table, population-weighted blend of NLD/FRA (59%/41%) used instead'
    key = tuple(sorted([a, b]))
    if key in raw_lut:
        return raw_lut[key], ''
    return 0.0, 'imputed: pair absent from raw table (no fallback rule applies), defaulted to 0'

Raw Melitz-Toubal country-pairs: 21,736


## 3. Build the complete, closed lookup table

In [4]:
rows = []
n_fallback = 0
for a, b in combinations(isos, 2):
    val, note = resolve_prox1(a, b)
    if note:
        n_fallback += 1
    rows.append({'iso3_a': a, 'iso3_b': b, 'prox1': val, 'note': note})
for a in isos:
    rows.append({'iso3_a': a, 'iso3_b': a, 'prox1': 1.0, 'note': ''})

pairs_final = pd.DataFrame(rows).sort_values(['iso3_a', 'iso3_b']).reset_index(drop=True)
print(f'Pairs exported: {len(pairs_final):,} ({len(isos)} countries, self-pairs included)')
print(f'Pairs resolved via fallback (not in raw table): {n_fallback}')
print(f'prox1 range: [{pairs_final["prox1"].min():.3f}, {pairs_final["prox1"].max():.3f}]')
assert pairs_final['prox1'].between(0, 1).all(), 'prox1 must be within [0,1]'
assert pairs_final['prox1'].notna().all(), 'prox1 must have no missing values'
print()
print('Breakdown of imputed/adjusted rows by reason:')
print(pairs_final.loc[pairs_final['note'] != '', 'note'].value_counts().to_string())

Pairs exported: 2,080 (64 countries, self-pairs included)
Pairs resolved via fallback (not in raw table): 186
prox1 range: [0.000, 1.000]

Breakdown of imputed/adjusted rows by reason:
note
imputed: Belgium absent from raw table, population-weighted blend of NLD/FRA (59%/41%) used instead    61
imputed: pair absent from raw table (no fallback rule applies), defaulted to 0                         61
imputed: Monaco value stands in for France (Monaco absent from raw table)                              58
imputed: Monaco set to 1.0 (French-official country pair)                                               4
imputed: Belgium fallback -- NLD/FRA values also missing, defaulted to 0                                1
imputed: Monaco fallback -- France's own value also missing, defaulted to 0                             1


## 4. Save

In [5]:
pairs_final.to_csv(OUT_PATH, index=False)
print(f'Saved {len(pairs_final)} rows -> {OUT_PATH}')

Saved 2080 rows -> C:\Users\ALESSANDRO\Documents\GitHub\tennis-homophily\data\gravity\ling_prox_pairs_final.csv
